# Reading datasets into BDF

BDF supports a few ways to read source data:

- a URL for a file hosted on the web
- a path for a local file
- an identifier from the BDF data registry

In this notebook, we will show some examples for how to read data from these different sources. We will also demonstrate some helpful features including:

- viewing the raw data in the format supplied from the vendor / original source
- keeping only the required columns or including all columns

In [ ]:
# Import the package
import bdf

In [ ]:
# Read from a local file path
filepath = "../../data/SINTEF__LiGrR2032__2024-04-30__25degC__Landt.csv"

# read() returns (frame, metadata); pass lazy=False to get a materialised polars DataFrame
df, meta = bdf.read(filepath, lazy=False)
df.head()

In [ ]:
# Peek at the raw vendor file without normalization to BDF
vendor_df, _ = bdf.read(filepath, normalize=False, validate=False, lazy=False)
vendor_df.head()

In [ ]:
# Keep only the BDF required columns
df_req, _ = bdf.read(filepath, include_optional=False, lazy=False)
print(df_req.columns)

# Include all the columns from the raw data that have a BDF equivalent
df_all, _ = bdf.read(filepath, include_optional=True, lazy=False)
print(df_all.columns[:12])

In [ ]:
# Read data on the Web from a URL
df, meta = bdf.read("https://zenodo.org/records/17289383/files/SINTEF__NaCR32140-MP10-04__2025-08-25__GITT_0p05C_25degC__BioLogic.mpt", lazy=False)
df.head()

## Read from the dataset registry

The dataset registry is a JSON file (datasets.json) that maps ids to URLs and
optional plugin hints. This is separate from the metadata registry built by
`bdf.build_registry`. Resolve an id to its entry with `bdf.load_registry` /
`bdf.get_entry`, then pass the entry's URL to `bdf.read`.

In [ ]:
# Read from the BDF data registry
# This registry is a local JSON file (datasets.json), separate from the metadata registry.
import json
from pathlib import Path

registry_path = Path("out/datasets.json")
registry_path.parent.mkdir(parents=True, exist_ok=True)
registry_path.write_text(json.dumps({
    "schema_version": "0.3",
    "entries": [
        {
            "id": "sintef-biologic-demo",
            "url": "https://zenodo.org/records/17289383/files/SINTEF__NaCR32140-MP10-04__2025-08-25__GITT_0p05C_25degC__BioLogic.mpt",
            "plugin": "biologic_mpt",
        }
    ],
}, indent=2), encoding="utf-8")

reg = bdf.load_registry(registry_path)
entry = bdf.get_entry(reg, "sintef-biologic-demo")
df, meta = bdf.read(entry["url"], lazy=False)
df.head()